# Notebook 1 de 3 — Fotos reales de autos

Proyecto **Detección de daños sintéticos en fotos de siniestros**. El dataset se construye desde cero con un criterio de diseño central: **todas las clases salen del mismo proceso**. Lo único que diferencia una foto auténtica de una manipulada es la manipulación; el tamaño, la calidad y el origen del archivo son iguales, para que el modelo no pueda apoyarse en esas diferencias.

**Estructura del pipeline (3 notebooks):**

1. **Este notebook:** obtiene fotos reales de autos (sanos y con daño real) y las lleva a un formato base común: 768×512 px, un solo auto bien visible y sin duplicados.
2. **Notebook 2:** genera las imágenes manipuladas a partir de esas fotos: (a) la misma foto con daño agregado por IA (*inpainting*), (b) la misma foto con una edición de *control* sin daño y (c) autos 100 % generados por IA.
3. **Notebook 3:** aplica a todas las imágenes el mismo paso final (misma compresión JPEG), controla que no queden atajos y arma el dataset final.

**Contenido de este notebook**

- Búsqueda y descarga de fotos de **Wikimedia Commons** con licencias abiertas, registrando autor y licencia de cada una.
- Descarte automático de las fotos que no cumplen los criterios: sin un auto claro, con el auto muy chico, con varios autos o duplicadas.
- Recorte y redimensionado a 768×512, y cálculo de la posición del auto (la usa el notebook 2).
- Revisión visual de las fotos aceptadas.
- Incorporación opcional de fotos propias tomadas con celular, las más parecidas a las de un siniestro real.

**Ejecución:** Google Colab, celdas en orden. La variable `PRUEBA_RAPIDA = True` descarga pocas fotos para verificar el funcionamiento; con `False` se arma el conjunto completo. Con GPU el detector de autos es más rápido.

> **Licencias:** se usan fotos de Commons con licencia CC0, dominio público, CC BY o CC BY-SA. Para un proyecto de estudio alcanza con registrar autor y licencia, como hace el notebook. Si el dataset o el modelo se publicaran o se usaran con fines comerciales, habría que revisar las condiciones de cada licencia.

## 1. Instalación e imports

In [12]:
%pip install -q requests pillow pillow-heif pandas matplotlib

In [13]:
import os, sys, re, io, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
try:                                    # permite leer fotos HEIC de iPhone
    from pillow_heif import register_heif_opener
    register_heif_opener()
except Exception:
    pass

import torch
import torchvision
from torchvision.transforms import functional as TF

print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)

PyTorch 2.11.0+cu128 | torchvision 0.26.0+cu128


## 2. Configuración

El único valor a completar es `USER_AGENT`: la política de uso de Wikimedia pide que las descargas se identifiquen con un mail de contacto.

- **Tamaño base 768×512** (proporción 3:2, como una foto de celular apaisada). Todo el proyecto usa este tamaño.
- Se piden a Commons **miniaturas de 1280 px de ancho**, más livianas que el original. Solo se aceptan fotos cuyo original tenga al menos 1600 px de ancho, para que conserven nitidez al reducirse.
- `CONSULTAS` contiene las búsquedas. Se busca el **modelo de auto en el nombre del archivo** (`intitle:"Toyota Corolla"`), que es la opción más precisa de Commons, y se excluyen títulos con palabras como "car park", "railway" o "museum". Como los resultados igual son irregulares, después se aplican filtros automáticos y una revisión visual.
- **Fotos con daño real:** Commons tiene pocas fotos de autos dañados con calidad suficiente, por lo que este grupo resulta chico. Las fuentes alternativas (fotos propias, dataset CarDD) se describen en la sección 4 y al final.

In [14]:
# ---------- Carpetas de trabajo ----------
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/deteccion-fraude-v2")
else:
    RAIZ = Path("./deteccion-fraude-v2")

CARPETA          = RAIZ / "01_reales"
DIR_ORIGINALES   = CARPETA / "originales"       # miniaturas descargadas (1280 px)
DIR_BASE         = CARPETA / "base"             # fotos ya estandarizadas 768x512 (PNG sin pérdida)
DIR_PROPIAS      = CARPETA / "fotos_propias"    # opcional: subcarpetas "sanas" y "danadas"
for d in [DIR_ORIGINALES, DIR_BASE, DIR_PROPIAS / "sanas", DIR_PROPIAS / "danadas"]:
    d.mkdir(parents=True, exist_ok=True)
MANIFEST_DESCARGAS = CARPETA / "manifest_descargas.csv"
MANIFEST_REALES    = CARPETA / "manifest_reales.csv"

# ---------- Identificación para Wikimedia (EDITAR) ----------
USER_AGENT = "deteccion-fraude-estudio/1.0 (contacto@ejemplo.com)"   # mail de contacto (lo pide la política de uso de Wikimedia)

# ---------- Formato base ----------
CANVAS_W, CANVAS_H = 768, 512
ANCHO_DESCARGA     = 1280        # ancho de la miniatura solicitada a Commons
MIN_ANCHO_ORIGINAL = 1600        # se descartan originales más chicos

# ---------- Cantidad objetivo ----------
OBJETIVO = {"sano": 300, "danado": 100}
MAX_POR_CONSULTA = 120           # tope de resultados por búsqueda
MAX_ACEPTADAS_POR_CONSULTA = 25  # tope de fotos por búsqueda, para no concentrar todo en un mismo modelo

# ---------- Búsquedas en Commons ----------
# Búsqueda por MODELO dentro del NOMBRE del archivo (intitle:): casi todas las fotos de autos en Commons se
# llaman "Toyota Corolla E210 front.jpg". Buscar palabras sueltas como "car street" trae calles y estacionamientos.
# Los "-intitle:..." excluyen títulos con esas palabras.
EXCLUIR = ('-intitle:interior -intitle:engine -intitle:museum -intitle:show -intitle:railway -intitle:train '
           '-intitle:toy -intitle:model -intitle:"car park" -intitle:parking')
MODELOS = ["Toyota Corolla", "Volkswagen Golf", "Ford Focus", "Honda Civic", "Renault Clio", "Peugeot 208",
           "Fiat Punto", "Volkswagen Gol", "Renault Sandero", "Chevrolet Onix", "Toyota Hilux", "Ford Fiesta",
           "Opel Astra", "Hyundai i20", "Nissan Micra", "Kia Rio", "Skoda Octavia", "Mazda 3"]
CONSULTAS = {
    "sano": [f'intitle:"{m}" {EXCLUIR}' for m in MODELOS],
    "danado": [f'intitle:"{t}" {EXCLUIR}' for t in
               ["car crash", "car accident", "damaged car", "crashed car", "car collision", "wrecked car",
                "hail damage", "dented", "traffic collision", "accident damage"]],
}
# Si el título contiene alguna de estas palabras, la foto se descarta sin descargarla
TITULOS_EXCLUIDOS = (r"car park|parking|railway|train\b|tank car|interior|engine|museum|toy|lego|diecast|drawing|painting|logo|"
                     r"wheel|steering|dashboard|riot|unrest|protest|demonstration|bomb|explosion|war\b|attack|burnt|burned|fire\b")

# ---------- Reinicio ----------
# True: borra lo descargado y procesado antes (no toca fotos_propias) y vuelve a descargar todo.
REINICIAR = False

# ---------- Filtros de calidad ----------
MIN_PUNTAJE_AUTO   = 0.70        # confianza mínima del detector para contar un auto
MIN_FRAC_AUTO      = 0.20        # el auto debe ocupar al menos 20 % de la foto
MIN_FRAC_EN_RECORTE = 0.80       # y quedar mayormente dentro del recorte 3:2 (un auto apenas cortado es realista)
UMBRAL_DUPLICADO   = 6           # distancia de hash: <= 6 se considera la misma foto

# ---------- Modo prueba ----------
PRUEBA_RAPIDA = False
if PRUEBA_RAPIDA:
    OBJETIVO = {"sano": 12, "danado": 6}          # se piden de más: los filtros descartan cerca de la mitad
    MAX_POR_CONSULTA = 12
    MAX_ACEPTADAS_POR_CONSULTA = 3                # 3 fotos por modelo -> salen de 4 modelos distintos
    CONSULTAS = {"sano": CONSULTAS["sano"][:4], "danado": CONSULTAS["danado"][:4]}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA)
print("Dispositivo:", DEVICE, "| carpeta de trabajo:", CARPETA.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cuda | carpeta de trabajo: /content/drive/MyDrive/deteccion-fraude-v2/01_reales


## 3. Búsqueda y descarga desde Wikimedia Commons

La API de Commons se puede consultar sin cuenta. Por cada archivo se obtienen la dirección de descarga de la miniatura, el tamaño original, el tipo de archivo, la **licencia** y el **autor**. Se conservan solo:

- fotos JPEG cuyo original tenga al menos `MIN_ANCHO_ORIGINAL` px de ancho;
- licencias que permiten obras derivadas (CC0, dominio público, CC BY, CC BY-SA). Se descartan las **NC** (no comercial) y las **ND** (sin obras derivadas): como las fotos se editan con IA, las ND no se pueden usar.

Cada foto queda registrada en `manifest_descargas.csv`. Si la ejecución se interrumpe, al volver a correr la celda el proceso **retoma** desde donde quedó, sin descargar dos veces la misma foto.

In [15]:
API = "https://commons.wikimedia.org/w/api.php"
SESION = requests.Session()
SESION.headers.update({"User-Agent": USER_AGENT})

def api_get(params, reintentos=5):
    # Consulta a la API de Commons con reintentos y espera ante límites de frecuencia (429).
    p = {"format": "json", "formatversion": "2", **params}
    for intento in range(reintentos):
        r = SESION.get(API, params=p, timeout=60)
        if r.status_code == 200:
            return r.json()
        time.sleep(2 * (intento + 1))
    r.raise_for_status()

def descargar_bytes(url, reintentos=5):
    for intento in range(reintentos):
        r = SESION.get(url, timeout=120)
        if r.status_code == 200:
            return r.content
        time.sleep(3 * (intento + 1))
    r.raise_for_status()

def licencia_ok(texto):
    t = (texto or "").lower().strip()
    partes = re.split(r"[\s\-_/]+", t)
    if "nc" in partes or "nd" in partes:
        return False
    return t.startswith(("cc by", "cc-by", "cc0", "public domain", "pd"))

def limpiar_html(texto):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", texto or "")).strip()

def buscar_commons(consulta, max_resultados):
    params = {
        "action": "query", "generator": "search",
        "gsrsearch": f"{consulta} filetype:bitmap", "gsrnamespace": 6, "gsrlimit": 50,
        "prop": "imageinfo", "iiprop": "url|size|mime|extmetadata", "iiurlwidth": ANCHO_DESCARGA,
        "iiextmetadatafilter": "LicenseShortName|Artist|LicenseUrl",
    }
    resultados = []
    while len(resultados) < max_resultados:
        resp = api_get(params)
        for pg in resp.get("query", {}).get("pages", []):
            info = (pg.get("imageinfo") or [None])[0]
            if info:
                resultados.append({"titulo": pg["title"], "info": info})
        if "continue" in resp:
            params.update(resp["continue"])
        else:
            break
    return resultados[:max_resultados]

def es_valida(info, titulo=""):
    meta = info.get("extmetadata", {})
    lic = meta.get("LicenseShortName", {}).get("value", "")
    if re.search(TITULOS_EXCLUIDOS, titulo, flags=re.IGNORECASE):
        return False
    return (info.get("mime") == "image/jpeg"
            and info.get("width", 0) >= MIN_ANCHO_ORIGINAL
            and bool(info.get("thumburl"))
            and licencia_ok(lic))

In [16]:
if REINICIAR:
    for p in list(DIR_ORIGINALES.glob("*")) + list(DIR_BASE.glob("*")):
        p.unlink()
    for p in [MANIFEST_DESCARGAS, MANIFEST_REALES]:
        if p.exists():
            p.unlink()
    print("REINICIAR = True: se borró lo descargado y procesado antes (fotos_propias no se toca).")
    print("Con REINICIAR = True, cada nueva ejecución de esta celda vuelve a borrar lo descargado.\n")

COLUMNAS_DESC = ["id", "tipo", "fuente", "titulo", "url_pagina", "autor", "licencia",
                 "ancho_orig", "alto_orig", "consulta", "ruta_local"]

if MANIFEST_DESCARGAS.exists():
    df_desc = pd.read_csv(MANIFEST_DESCARGAS)
else:
    df_desc = pd.DataFrame(columns=COLUMNAS_DESC)

titulos_vistos = set(df_desc["titulo"].dropna())
contador = {"sano": 0, "danado": 0}
for _, f in df_desc[df_desc["fuente"] == "commons"].iterrows():
    contador[f["tipo"]] += 1

def nuevo_id(tipo):
    prefijo = "r" if tipo == "sano" else "d"
    return f"{prefijo}{contador[tipo] + 1:04d}"

nuevas = []
for tipo, consultas in CONSULTAS.items():
    for consulta in consultas:
        if contador[tipo] >= OBJETIVO[tipo]:
            break
        print(f"[{tipo}] buscando: {consulta!r}")
        de_esta_consulta = 0
        for res in buscar_commons(consulta, MAX_POR_CONSULTA):
            if contador[tipo] >= OBJETIVO[tipo] or de_esta_consulta >= MAX_ACEPTADAS_POR_CONSULTA:
                break
            info, titulo = res["info"], res["titulo"]
            if titulo in titulos_vistos or not es_valida(info, titulo):
                continue
            try:
                datos = descargar_bytes(info["thumburl"])
            except Exception as e:
                print("   no se pudo bajar", titulo, e)
                continue
            id_ = nuevo_id(tipo)
            ruta = DIR_ORIGINALES / f"{id_}.jpg"
            ruta.write_bytes(datos)
            meta = info.get("extmetadata", {})
            fila = {"id": id_, "tipo": tipo, "fuente": "commons", "titulo": titulo,
                    "url_pagina": info.get("descriptionurl", ""),
                    "autor": limpiar_html(meta.get("Artist", {}).get("value", "")),
                    "licencia": meta.get("LicenseShortName", {}).get("value", ""),
                    "ancho_orig": info.get("width"), "alto_orig": info.get("height"),
                    "consulta": consulta, "ruta_local": str(ruta)}
            nuevas.append(fila)
            titulos_vistos.add(titulo)
            contador[tipo] += 1
            de_esta_consulta += 1
            time.sleep(0.5)            # pausa entre descargas para no sobrecargar el servidor
        # se guarda después de cada búsqueda por si la ejecución se interrumpe
        if nuevas:
            df_desc = pd.concat([df_desc, pd.DataFrame(nuevas)], ignore_index=True)
            df_desc.to_csv(MANIFEST_DESCARGAS, index=False)
            nuevas = []

print("\nDescargadas hasta ahora:", contador)


Descargadas hasta ahora: {'sano': 300, 'danado': 100}


## 4. Fotos propias (opcional)

Las fotos de celular de autos reales son las más parecidas a las que llegan en un reclamo de siniestro. Pueden incorporarse fotos de autos (tomadas con permiso, preferentemente sin que se vea la patente) en dos carpetas dentro de `deteccion-fraude-v2/01_reales/fotos_propias/`:

- `sanas/`: autos sin daño.
- `danadas/`: autos con daño real.

Estas fotos pasan por los mismos filtros que las de Commons. Si las carpetas están vacías, la celda no agrega nada.

In [17]:
def registrar_fotos_propias(df):
    ya = set(df["ruta_local"].dropna())
    filas, cont = [], {"sano": 0, "danado": 0}
    for _, f in df[df["fuente"] == "propia"].iterrows():
        cont[f["tipo"]] += 1
    for tipo, sub in [("sano", "sanas"), ("danado", "danadas")]:
        for p in sorted((DIR_PROPIAS / sub).glob("*")):
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp", ".heic"} or str(p) in ya:
                continue
            cont[tipo] += 1
            pref = "rp" if tipo == "sano" else "dp"
            filas.append({"id": f"{pref}{cont[tipo]:03d}", "tipo": tipo, "fuente": "propia",
                          "titulo": p.name, "url_pagina": "", "autor": "propia", "licencia": "propia",
                          "ancho_orig": None, "alto_orig": None, "consulta": "", "ruta_local": str(p)})
    if filas:
        df = pd.concat([df, pd.DataFrame(filas)], ignore_index=True)
        df.to_csv(MANIFEST_DESCARGAS, index=False)
    return df, len(filas)

df_desc, n_propias = registrar_fotos_propias(df_desc)
print(f"Fotos propias nuevas registradas: {n_propias}")
print(df_desc.groupby(["fuente", "tipo"]).size().to_string())

Fotos propias nuevas registradas: 0
fuente   tipo  
commons  danado    100
         sano      300


## 4b. Fotos de datasets de Kaggle (opcional)

Sirve para sumar **fotos con daño real**, el grupo más chico. Las fotos se bajan de Kaggle directo a Colab (sin pasar por la compu), se copian a `01_reales/fotos_kaggle/` en Drive y después pasan por **los mismos filtros** que las de Commons: auto entero y visible, tamaño mínimo, duplicados y revisión visual.

**Antes de ejecutar (una sola vez):**

1. En Kaggle: *Settings → API Tokens*, escribir un nombre (por ejemplo `colab-danos`) y tocar *Generate*. Copiar el token: Kaggle lo muestra una sola vez.
2. En Colab, en el panel izquierdo, el ícono de la **llave** (*Secretos*): agregar `KAGGLE_API_TOKEN` con ese valor y activar "Acceso al notebook". Así el token no queda escrito en el notebook.

(Con credenciales viejas de `kaggle.json` también funciona: secretos `KAGGLE_USERNAME` y `KAGGLE_KEY`.)

**Qué datasets se usan:** se configuran en `DATASETS_KAGGLE`. Para cada uno se indica el tipo (`danado` o `sano`), un texto que tiene que aparecer en la ruta del archivo (para quedarse solo con la carpeta de daños) y la licencia, que queda registrada en `atribuciones.csv`.

- `humansintheloop/car-parts-and-car-damages`: licencia **CC0** (dominio público). **Resultado de la prueba (24/10/2026):** la carpeta de daños tiene 998 fotos de entre 570 y 916 px de ancho (mediana 736): ninguna alcanza el mínimo y casi ninguna llega a 768 px. Agregarlas igual haría que las fotos con daño real sean las más chicas, el mismo atajo que invalidó la primera versión del proyecto. Por eso `USAR_KAGGLE` queda en `False`.
- Antes de agregar otro dataset, revisar su licencia en la pestaña *License* de Kaggle y que las fotos no sean miniaturas.

Se copian solo las fotos de al menos `MIN_ANCHO_KAGGLE` px de ancho: más chicas no sirven para el análisis forense. Muchas fotos de daño son primeros planos (una puerta, un paragolpes) y el filtro de la sección 6 las va a descartar por "auto cortado": es esperable.

In [ ]:
# ---------- Configuración ----------
USAR_KAGGLE      = False         # ver nota: el dataset configurado resultó tener fotos demasiado chicas
DIR_KAGGLE       = CARPETA / "fotos_kaggle"
MIN_ANCHO_KAGGLE = 1024          # px de ancho mínimos de la foto original
MAX_POR_DATASET  = 400           # tope de fotos copiadas por dataset
DATASETS_KAGGLE = [
    # (dataset de Kaggle, tipo, texto que debe aparecer en la ruta ("" = todas), licencia)
    ("humansintheloop/car-parts-and-car-damages", "danado", "Car damages dataset/", "CC0 1.0"),
]
EXCLUIR_RUTA = ["mask", "annot", "label", "/ann/", "segment"]   # máscaras y anotaciones, no son fotos
EXTENSIONES  = {".jpg", ".jpeg", ".png", ".webp"}

def conectar_kaggle():
    """Lee el token de los Secretos de Colab; si no está, kagglehub lo pide en pantalla."""
    if EN_COLAB:
        from google.colab import userdata
        for nombre in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
            try:
                os.environ[nombre] = userdata.get(nombre)
            except Exception:
                pass
    import kagglehub
    if not (os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))):
        kagglehub.login()
    return kagglehub

def fotos_de_dataset(carpeta, filtro):
    for p in sorted(Path(carpeta).rglob("*")):
        ruta = str(p).lower().replace("\\", "/")
        if p.suffix.lower() not in EXTENSIONES or any(x in ruta for x in EXCLUIR_RUTA):
            continue
        if filtro and filtro.lower() not in p.relative_to(carpeta).parent.as_posix().lower() + "/":
            continue
        yield p

def registrar_fotos_kaggle(df, descargar):
    ya = set(df["titulo"].dropna())
    cont = {"sano": 0, "danado": 0}
    for _, f in df[df["fuente"] == "kaggle"].iterrows():
        cont[f["tipo"]] += 1
    filas = []
    for dataset, tipo, filtro, licencia in DATASETS_KAGGLE:
        carpeta = Path(descargar(dataset))
        destino = DIR_KAGGLE / dataset.replace("/", "__")
        destino.mkdir(parents=True, exist_ok=True)
        candidatas = list(fotos_de_dataset(carpeta, filtro))
        motivos = {"chica": 0, "ya_registrada": 0, "ilegible": 0, "copiada": 0}
        copiadas = 0
        for p in candidatas:
            if copiadas >= MAX_POR_DATASET:
                break
            titulo = f"{dataset}:{p.relative_to(carpeta).as_posix()}"
            if titulo in ya:
                motivos["ya_registrada"] += 1
                continue
            try:
                with Image.open(p) as im:
                    ancho, alto = ImageOps.exif_transpose(im).size
            except Exception:
                motivos["ilegible"] += 1
                continue
            if ancho < MIN_ANCHO_KAGGLE:
                motivos["chica"] += 1
                continue
            cont[tipo] += 1
            id_ = f"{'rk' if tipo == 'sano' else 'dk'}{cont[tipo]:04d}"
            ruta = destino / f"{id_}{p.suffix.lower()}"
            ruta.write_bytes(p.read_bytes())
            filas.append({"id": id_, "tipo": tipo, "fuente": "kaggle", "titulo": titulo,
                          "url_pagina": f"https://www.kaggle.com/datasets/{dataset}",
                          "autor": dataset.split("/")[0], "licencia": licencia,
                          "ancho_orig": ancho, "alto_orig": alto, "consulta": dataset, "ruta_local": str(ruta)})
            ya.add(titulo)
            copiadas += 1
            motivos["copiada"] += 1
        print(f"[{dataset}] {len(candidatas)} fotos que cumplen el filtro de ruta | {motivos}")
        if len(candidatas) == 0:
            print("   ATENCIÓN: ninguna foto coincidió con el filtro. Carpetas del dataset:")
            for d in sorted({q.parent.relative_to(carpeta).as_posix() for q in Path(carpeta).rglob('*') if q.is_file()})[:20]:
                print("    ", d)
    if filas:
        df = pd.concat([df, pd.DataFrame(filas)], ignore_index=True)
        df.to_csv(MANIFEST_DESCARGAS, index=False)
    return df, len(filas)

if USAR_KAGGLE:
    %pip install -q -U "kagglehub>=0.4.1"
    kh = conectar_kaggle()
    df_desc, n_kaggle = registrar_fotos_kaggle(df_desc, kh.dataset_download)
    print(f"\nFotos de Kaggle nuevas registradas: {n_kaggle}")
    print(df_desc.groupby(["fuente", "tipo"]).size().to_string())
else:
    print("USAR_KAGGLE = False: no se agregan fotos de Kaggle.")

## 5. Detector de autos y funciones de estandarización

Para ubicar **dónde está el auto** se usa un detector preentrenado (Faster R-CNN de torchvision, entrenado con el dataset COCO, que incluye la clase "car"). Con él:

- se conservan solo las fotos con **un auto principal claro**, que ocupe una parte importante de la imagen y sin otro auto de tamaño parecido al lado;
- se recorta a proporción 3:2 **centrando en el auto**, y se descarta la foto si el recorte lo corta demasiado;
- se guarda la **caja del auto** (`caja_x0…caja_y1`, en coordenadas de 768×512). El notebook 2 la usa para editar solamente zonas de la carrocería.

Los **duplicados** se detectan con un *hash* perceptual: una huella de 64 bits de la imagen, en la que dos fotos casi iguales producen huellas casi iguales.

In [18]:
# [CARGA_MODELO] Detector de autos (se descarga una vez, ~160 MB)
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights

_PESOS = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn_v2(weights=_PESOS, box_score_thresh=0.5).eval().to(DEVICE)
ID_AUTO = _PESOS.meta["categories"].index("car")
ID_CAMION = _PESOS.meta["categories"].index("truck")

@torch.no_grad()
def detectar_autos(img):
    # Devuelve lista de cajas [x0, y0, x1, y1] de autos/camionetas con buena confianza.
    t = TF.to_tensor(img).to(DEVICE)
    out = detector([t])[0]
    cajas = []
    for b, l, s in zip(out["boxes"].cpu().numpy(), out["labels"].cpu().numpy(), out["scores"].cpu().numpy()):
        if int(l) in (ID_AUTO, ID_CAMION) and float(s) >= MIN_PUNTAJE_AUTO:
            cajas.append(b)
    return cajas

In [19]:
def area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])

def auto_principal(img):
    # Devuelve la caja del auto principal o (None, motivo) si la foto no sirve.
    cajas = detectar_autos(img)
    if not cajas:
        return None, "sin_auto"
    areas = [area(b) for b in cajas]
    orden = np.argsort(areas)[::-1]
    principal = cajas[orden[0]]
    if areas[orden[0]] / (img.width * img.height) < MIN_FRAC_AUTO:
        return None, "auto_muy_chico"
    if len(cajas) > 1 and areas[orden[1]] > 0.5 * areas[orden[0]]:
        return None, "varios_autos"
    return principal, "ok"

def recorte_3_2(img, caja):
    # Ventana 3:2 lo más grande posible, centrada en el auto. Devuelve (ventana, fracción del auto adentro).
    W, H = img.size
    asp = CANVAS_W / CANVAS_H
    if W / H >= asp:
        ch, cw = H, int(round(H * asp))
    else:
        cw, ch = W, int(round(W / asp))
    cx, cy = (caja[0] + caja[2]) / 2, (caja[1] + caja[3]) / 2
    x0 = int(round(min(max(cx - cw / 2, 0), W - cw)))
    y0 = int(round(min(max(cy - ch / 2, 0), H - ch)))
    ventana = (x0, y0, x0 + cw, y0 + ch)
    inter = area((max(caja[0], ventana[0]), max(caja[1], ventana[1]),
                  min(caja[2], ventana[2]), min(caja[3], ventana[3])))
    return ventana, inter / max(area(caja), 1e-6)

def dhash(img, n=8):
    g = img.convert("L").resize((n + 1, n), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

def estandarizar(ruta):
    # Devuelve (imagen 768x512, caja en coords base, hash, motivo). Si no sirve: imagen=None.
    img = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    caja, motivo = auto_principal(img)
    if caja is None:
        return None, None, None, motivo
    ventana, frac = recorte_3_2(img, caja)
    if frac < MIN_FRAC_EN_RECORTE:
        return None, None, None, "auto_cortado"
    cw = ventana[2] - ventana[0]
    if cw < CANVAS_W:
        return None, None, None, "foto_chica"
    recorte = img.crop(ventana).resize((CANVAS_W, CANVAS_H), Image.LANCZOS)
    k = CANVAS_W / cw
    caja_base = [float(np.clip((caja[0] - ventana[0]) * k, 0, CANVAS_W)),
                 float(np.clip((caja[1] - ventana[1]) * k, 0, CANVAS_H)),
                 float(np.clip((caja[2] - ventana[0]) * k, 0, CANVAS_W)),
                 float(np.clip((caja[3] - ventana[1]) * k, 0, CANVAS_H))]
    return recorte, caja_base, dhash(recorte), "ok"

## 6. Procesamiento de las fotos descargadas

Para cada foto se detecta el auto, se recorta, se redimensiona a 768×512 y se guarda como **PNG**, un formato sin pérdida: así no se agrega una compresión intermedia, y la compresión final se aplica una sola vez, igual para todas, en el notebook 3. Las fotos que no pasan los filtros quedan registradas con el motivo en la columna `estado`.

In [20]:
hashes_vistos = []            # huellas de las fotos ya aceptadas
filas = []
for i, f in df_desc.iterrows():
    ruta_base = DIR_BASE / f"{f['id']}.png"
    try:
        img_b, caja, h, motivo = estandarizar(f["ruta_local"])
    except Exception as e:
        img_b, caja, h, motivo = None, None, None, f"error_lectura"
    if img_b is not None:
        if any(int((h != h2).sum()) <= UMBRAL_DUPLICADO for h2 in hashes_vistos):
            img_b, motivo = None, "duplicada"
        else:
            hashes_vistos.append(h)
            img_b.save(ruta_base)
    fila = f.to_dict()
    fila.update({"estado": motivo,
                 "caja_x0": caja[0] if caja else None, "caja_y0": caja[1] if caja else None,
                 "caja_x1": caja[2] if caja else None, "caja_y1": caja[3] if caja else None,
                 "ruta_base": str(ruta_base) if img_b is not None else ""})
    filas.append(fila)

df_proc = pd.DataFrame(filas)
print("Resultado del filtrado automático:")
print(pd.crosstab(df_proc["tipo"], df_proc["estado"]).to_string())
print("\nDetalle (primeras 40 filas):")
print(df_proc[["id", "tipo", "titulo", "estado"]].head(40).to_string())
if (df_proc["estado"] == "ok").sum() == 0:
    print("\nNinguna foto pasó el filtro. Si en la columna 'estado' predominan 'sin_auto' o 'auto_muy_chico', "
          "las búsquedas están devolviendo escenas de calle en lugar de autos (ver CONSULTAS y TITULOS_EXCLUIDOS).")

Resultado del filtrado automático:
estado  auto_cortado  auto_muy_chico  duplicada   ok  sin_auto  varios_autos
tipo                                                                        
danado             1              32          1   38        23             5
sano              15              13          1  245         7            19

Detalle (primeras 40 filas):
       id    tipo                                                                           titulo          estado
0   r0001    sano                             File:2018 Volkswagen Golf R-Line TSi EVO S-A 1.5.jpg              ok
1   r0002    sano                                     File:2020 Volkswagen Golf Style 1.5 Side.jpg    auto_cortado
2   r0003    sano                                     File:2020 Volkswagen Golf Style 1.5 Rear.jpg    varios_autos
3   r0004    sano                                                  File:'05-'07 Ford Focus ZXW.jpg              ok
4   r0005    sano                              File:

## 7. Revisión visual

Los filtros automáticos no detectan todos los problemas. Las hojas de contacto muestran 30 fotos por vez, con su `id`, para identificar las que no corresponden (pinturas, maquetas, autos de carrera, fotos con mucha gente o mal clasificadas como sanas o dañadas). Los `id` descartados se registran en la celda siguiente.

In [21]:
def hoja_contacto(tipo, pagina=0, por_hoja=30, columnas=6):
    sub = df_proc[(df_proc["estado"] == "ok") & (df_proc["tipo"] == tipo)].reset_index(drop=True)
    chunk = sub.iloc[pagina * por_hoja:(pagina + 1) * por_hoja]
    filas_n = int(np.ceil(max(len(chunk), 1) / columnas))
    fig, ax = plt.subplots(filas_n, columnas, figsize=(columnas * 2.6, filas_n * 2.0))
    ax = np.atleast_1d(ax).flatten()
    for a in ax:
        a.axis("off")
    for a, (_, f) in zip(ax, chunk.iterrows()):
        a.imshow(Image.open(f["ruta_base"]))
        a.set_title(f["id"], fontsize=9)
    plt.suptitle(f"{tipo.upper()} - página {pagina} (de {int(np.ceil(len(sub) / por_hoja))})", y=1.0)
    plt.tight_layout(); plt.show()

hoja_contacto("sano", 0)

Output hidden; open in https://colab.research.google.com to view.

In [22]:
# El segundo argumento es el número de página (0, 1, 2, ...)
hoja_contacto("danado", 0)

Output hidden; open in https://colab.research.google.com to view.

In [23]:
# Ids descartados en la revisión visual y correcciones de tipo
DESCARTAR = ["r0005"]
CORREGIR_TIPO = {
    # "r0012": "danado",      # estaba como sana pero tiene daño
}
print("A descartar:", DESCARTAR, "| a corregir:", CORREGIR_TIPO)

A descartar: ['r0005'] | a corregir: {}


## 8. Resultado final: `manifest_reales.csv`

Se guardan solo las fotos aceptadas y no descartadas en la revisión visual. Este archivo es la entrada del notebook 2. Además se genera `atribuciones.csv`, con autor, licencia y enlace de cada foto de Commons.

In [24]:
df_ok = df_proc[(df_proc["estado"] == "ok") & (~df_proc["id"].isin(DESCARTAR))].copy()
for id_, nuevo_tipo in CORREGIR_TIPO.items():
    df_ok.loc[df_ok["id"] == id_, "tipo"] = nuevo_tipo

columnas_final = ["id", "tipo", "fuente", "ruta_base", "caja_x0", "caja_y0", "caja_x1", "caja_y1",
                  "titulo", "url_pagina", "autor", "licencia"]
df_ok[columnas_final].to_csv(MANIFEST_REALES, index=False)
df_ok[df_ok["fuente"] != "propia"][["id", "fuente", "titulo", "url_pagina", "autor", "licencia"]].to_csv(
    CARPETA / "atribuciones.csv", index=False)

print("Fotos reales aceptadas:")
print(df_ok.groupby(["fuente", "tipo"]).size().to_string())
print("\nLicencias:", df_ok["licencia"].value_counts().to_dict())

if (df_ok["tipo"] == "sano").sum() < 150:
    print("\nATENCIÓN: hay menos de 150 fotos sanas; para entrenar se apunta a 300 o más.")
if (df_ok["tipo"] == "danado").sum() < 50:
    print("ATENCIÓN: pocas fotos con daño real. Son necesarias en la clase auténtica para que el modelo "
          "no asocie la presencia de daño con la manipulación.")
print("\nGuardado en:", MANIFEST_REALES)

Fotos reales aceptadas:
fuente   tipo  
commons  danado     38
         sano      244

Licencias: {'CC BY-SA 4.0': 134, 'CC BY-SA 2.0': 37, 'Public domain': 28, 'CC BY-SA 3.0': 24, 'CC BY 2.0': 18, 'CC BY 3.0': 11, 'CC BY 4.0': 10, 'CC0': 8, 'CC BY-SA 3.0 it': 4, 'CC BY-SA 3.0 de': 3, 'CC BY-SA 2.0 de': 2, 'CC BY-SA 3.0 gr': 1, 'CC BY 2.5': 1, 'CC BY-SA 2.5': 1}
ATENCIÓN: pocas fotos con daño real. Son necesarias en la clase auténtica para que el modelo no asocie la presencia de daño con la manipulación.

Guardado en: /content/drive/MyDrive/deteccion-fraude-v2/01_reales/manifest_reales.csv


## 9. Resultado y próximos pasos

- Se obtuvieron **244 fotos sanas y 38 con daño real**. La cantidad de fotos se puede ampliar sumando modelos a `MODELOS` y volviendo a ejecutar desde la sección 3, que retoma sin repetir descargas.
- El grupo de fotos con daño real es chico, algo esperable en Commons. Se puede ampliar con fotos propias en `fotos_propias/danadas/`, con datasets de Kaggle (sección 4b) o con el dataset CarDD (4.000 fotos de alta resolución con daño real, que requiere un formulario de licencia).
- El siguiente paso es el **notebook 2**, que toma `manifest_reales.csv` y genera las imágenes manipuladas.